# Problem 1 — Find the Fault (Grover's Search)

One fault is hidden among `N` monitored nodes. Build Grover's search to find it in about √N steps instead of N.

**Your job:** fill in the two `TODO` functions (the oracle and the diffusion operator), run the search on a simulator for N = 4, 8, 16, then run it once on real hardware.

Deliverables map to Parts 1–4 in the problem packet. Answer the written Part in your response template.


In [ ]:
# Setup — these come preinstalled in the qBraid-SDK environment
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()


In [ ]:
# Problem parameters
N = 4                       # number of nodes (try 4, then 8, then 16)
n = int(np.log2(N))         # number of qubits
marked_node = 3             # which node is faulted (0..N-1); you can change this

# Optimal number of Grover iterations
iterations = int(np.floor((np.pi / 4) * np.sqrt(N)))
print(f"N={N}, qubits={n}, marked={marked_node}, iterations={iterations}")


In [ ]:
def oracle(qc, qubits, marked):
    """Flip the PHASE of the |marked> basis state, leaving all others unchanged."""
    # TODO: Implement the phase oracle.
    # Hint: put an X on each qubit that is 0 in the binary form of `marked`,
    #       apply a multi-controlled Z, then undo those X gates.
    #       A multi-controlled Z on `n` qubits = H on the target,
    #       mcx(controls, target), H on the target.
    raise NotImplementedError("Implement the oracle")


In [ ]:
def diffusion(qc, qubits):
    """Reflect the state about its average amplitude (the Grover diffuser)."""
    # TODO: Implement the diffusion operator.
    # Hint: H on all qubits, X on all qubits, a multi-controlled Z,
    #       X on all qubits, H on all qubits.
    raise NotImplementedError("Implement the diffusion operator")


In [ ]:
# Assemble the full Grover circuit (works once the TODOs are filled in)
def build_grover(n, marked, iterations):
    qc = QuantumCircuit(n, n)
    qubits = list(range(n))
    qc.h(qubits)                       # equal superposition
    for _ in range(iterations):
        oracle(qc, qubits, marked)
        diffusion(qc, qubits)
    qc.measure(qubits, qubits)
    return qc

qc = build_grover(n, marked_node, iterations)
qc.draw("mpl")


In [ ]:
# Run on the simulator
tqc = transpile(qc, sim)
counts = sim.run(tqc, shots=1024).result().get_counts()

# The measured node with the highest count should be the fault
best = max(counts, key=counts.get)
print("Most likely node (binary):", best, "= node", int(best, 2))

plt.bar(counts.keys(), counts.values())
plt.title(f"Grover search, N={N}")
plt.xlabel("measured node"); plt.ylabel("counts"); plt.show()


### Running on real quantum hardware (qBraid)

The cell below is a template. **Device IDs and the qBraid runtime API can change, so confirm both against your qBraid workspace** (the Devices tab, or `qbraid.runtime`) before you rely on it. Develop on the simulator first; use a real QPU only for the hardware deliverable, and remember the one-hardware-run-per-team budget.


In [ ]:
# --- Submit to a real QPU via qBraid ---
# Confirm the import and device ID against your workspace before running.
#
# from qbraid.runtime import QbraidProvider
# provider = QbraidProvider()
# for d in provider.get_devices():
#     print(d.id, d.num_qubits)          # find a device + its qubit count
# device = provider.get_device("<qpu_device_id>")   # e.g. an IonQ device
# job = device.run(qc, shots=200)
# counts = job.result().data.get_counts()
# print(counts)


## What to submit
- **Part 1:** this notebook running correctly for N = 4, 8, 16, with a histogram peaking on the marked node each time.
- **Part 2:** the hardware histogram from the QPU cell.
- **Part 3:** a plot of checks vs N (classical vs Grover) and 2–3 sentences on the simulator-vs-hardware difference.
- **Part 4:** the written explanation, in your response template.
